# PRAGMA Phase 4 - Structured masking inspection

Fits a `MaskingPlanner` (section 8; ADR 0005) against the processor from `002_fit_processor.ipynb`, samples masks on real tokenized records, and visually inspects them to confirm the Phase 4 exit gate: **all masking tests pass, and visual inspection of sampled masked records confirms that targets are not visible.**

Covers all three mask sources (individual token 15%, whole event 10%, semantic key 10%, combined by union), the `[UNK]` input-dropout corruption path (label `-100`, excluded from loss — see ADR 0005's correction), and the `PragmaCollator` integration that turns a `MaskingPlanner` into a real `PragmaBatch`.

In [1]:
from pathlib import Path

import pandas as pd

from pragma.config import MaskingConfig
from pragma.data import ParquetShardStore, PragmaCollator, TokenizedRecordDataset
from pragma.masking import IGNORE_LABEL, MaskingPlanner, MaskSource
from pragma.processing import PragmaProcessor
from pragma.processing.special_tokens import SpecialTokens
from pragma.schema import SchemaRegistry

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

registry = SchemaRegistry.default()
processor = PragmaProcessor.load(REPO_ROOT / "data" / "processor", registry)
train_dataset = TokenizedRecordDataset.from_store(
    REPO_ROOT / "data" / "shards", ParquetShardStore(), split="train"
)
print(f"{len(train_dataset)} train records")

399 train records


## Build a planner at the paper's default probabilities

In [2]:
config = MaskingConfig()
print(config)
planner = MaskingPlanner.from_registry(registry, processor.key_vocab, config)
print(f"{len(planner.maskable_key_ids)} maskable key ids")

MaskingConfig(token_mask_prob=0.15, event_mask_prob=0.1, key_mask_prob=0.1, unk_dropout_frac=0.05, seed=20260101)
16 maskable key ids


## Sample one record with a non-trivial history and inspect it token by token

For every event value token: its semantic key, the original value id, which mask source(s) selected it (if any), the corrupted input id actually fed to the model, and its training label. A masked position's `input_id` must never equal `original_id` unless it wasn't selected at all.

In [3]:
record = next(r for r in train_dataset._records if len(r.events) >= 3)
plan = planner.plan_record(record)
special = SpecialTokens()

rows = []
flat_idx = 0
for event_idx, event in enumerate(record.events):
    for field in event.fields:
        for key_id, value_id in zip(field.key_ids, field.value_ids, strict=True):
            origin = MaskSource(plan.origin[flat_idx])
            rows.append(
                {
                    "event_idx": event_idx,
                    "key": processor.key_vocab.key_for(key_id),
                    "original_value_id": value_id,
                    "mask_origin": origin.name,
                    "input_value_id": plan.input_value_ids[flat_idx],
                    "label": plan.labels[flat_idx],
                }
            )
            flat_idx += 1

inspection_df = pd.DataFrame(rows)
inspection_df

,event_idx,key,original_value_id,mask_origin,input_value_id,label
0,0,amount,28,KEY,2,28
1,0,currency,48,NONE,48,-100
2,0,direction,50,NONE,50,-100
3,0,description,354,NONE,354,-100
4,0,description,352,NONE,352,-100
...,...,...,...,...,...,...
205,37,amount,41,TOKEN|KEY,2,41
206,37,currency,45,NONE,45,-100
207,37,direction,50,NONE,50,-100
208,37,description,384,NONE,384,-100


## Confirm no masked target leaks into the input

For every row where `mask_origin != NONE`, `input_value_id` must be `[MASK]` or `[UNK]` — never the original value — and `label` must be either the original value (`[MASK]` case) or `-100` (`[UNK]` input-dropout case, ADR 0005).

In [4]:
masked_rows = inspection_df[inspection_df["mask_origin"] != "NONE"]
unmasked_rows = inspection_df[inspection_df["mask_origin"] == "NONE"]

leaked = masked_rows[masked_rows["input_value_id"] == masked_rows["original_value_id"]]
print(f"masked positions: {len(masked_rows)}")
print(f"leaked targets (should be 0): {len(leaked)}")
assert leaked.empty

bad_inputs = masked_rows[
    ~masked_rows["input_value_id"].isin([special.MASK, special.UNK])
]
assert bad_inputs.empty, "every masked input must be [MASK] or [UNK]"

assert (unmasked_rows["label"] == IGNORE_LABEL).all(), "unselected positions must have label -100"

unk_rows = masked_rows[masked_rows["input_value_id"] == special.UNK]
assert (unk_rows["label"] == IGNORE_LABEL).all(), "[UNK] input-dropout must have label -100"
mask_rows = masked_rows[masked_rows["input_value_id"] == special.MASK]
assert (mask_rows["label"] == mask_rows["original_value_id"]).all()
print("no leakage, all label rules hold")

masked positions: 90
leaked targets (should be 0): 0
no leakage, all label rules hold


## Mask-source coverage over the whole train split

Share of eligible event value tokens selected by each source, and how much they overlap (a position can be selected by more than one source — ADR 0005's union combination).

In [5]:
all_origin: list[int] = []
for r in train_dataset._records:
    all_origin.extend(planner.plan_record(r).origin)

import torch

origin_tensor = torch.tensor(all_origin, dtype=torch.long)
n_total = origin_tensor.shape[0]

coverage = {
    "token": float((origin_tensor & MaskSource.TOKEN != 0).float().mean()),
    "event": float((origin_tensor & MaskSource.EVENT != 0).float().mean()),
    "key": float((origin_tensor & MaskSource.KEY != 0).float().mean()),
    "any (union)": float((origin_tensor != 0).float().mean()),
}
n_multi_source = int(
    ((origin_tensor & (origin_tensor - 1) != 0) & (origin_tensor != 0)).sum()
)
print(f"n_total event value tokens: {n_total}")
pd.Series(coverage, name="coverage_rate").to_frame()

n_total event value tokens: 432913


,coverage_rate
token,0.149076
event,0.100367
key,0.098408
any (union),0.310074


In [6]:
print(f"positions selected by more than one source: {n_multi_source} "
      f"({n_multi_source / n_total:.4%} of all tokens)")

positions selected by more than one source: 15718 (3.6308% of all tokens)


## `PragmaCollator` integration: masking flows through into `PragmaBatch`

Same exit-gate checks, now at the batch level that will actually feed the model.

In [7]:
collator = PragmaCollator(masking_planner=planner)
records = [train_dataset[i] for i in range(16) if len(train_dataset[i].events) > 0]
batch = collator(records)
batch.validate()

selected = batch.event_mask_origin != 0
inputs_at_selected = batch.event_value_ids[selected]
only_special = ((inputs_at_selected == special.MASK) | (inputs_at_selected == special.UNK)).all()
unselected_labels_ok = (batch.event_mlm_labels[~selected] == IGNORE_LABEL).all()

print(f"n_event_tokens: {batch.n_event_tokens}, n_selected: {int(selected.sum())}")
print(f"every selected input is [MASK] or [UNK]: {bool(only_special)}")
print(f"every unselected label is -100: {bool(unselected_labels_ok)}")
assert bool(only_special) and bool(unselected_labels_ok)

n_event_tokens: 394, n_selected: 122
every selected input is [MASK] or [UNK]: True
every unselected label is -100: True


## Determinism check: same epoch reproduces the same mask, next epoch differs

Section 16.3's requirement — masking must be reproducible under a fixed seed, and must change across epochs.

In [8]:
batch_epoch_0_again = collator(records)
collator.set_epoch(1)
batch_epoch_1 = collator(records)

same_epoch_identical = bool((batch.event_mask_origin == batch_epoch_0_again.event_mask_origin).all())
next_epoch_differs = not bool((batch.event_mask_origin == batch_epoch_1.event_mask_origin).all())
print(f"epoch 0 reproducible: {same_epoch_identical}")
print(f"epoch 1 differs from epoch 0: {next_epoch_differs}")
assert same_epoch_identical and next_epoch_differs

epoch 0 reproducible: True
epoch 1 differs from epoch 0: True
